# TUE Reimbursement

This notebook demonstrates goal-oriented alignment visualization for the TUE Reimbursment case study.

- Models are loaded in the notebook.
- Handmade test cases are defined inline based on the traces discussed in the paper.
- Target rows are computed from the loaded goal model using `goal_model.compute_target_sets(target)`.
- Rendering is delegated to reusable helper functions in `Ui.goccva_ui`.



## Some notes from Hubert.
- This is the newer part
    - I have now updated the petri net to use the activities mentioned in the log
    - I have added a dependency between payment handled and a goal in the employee Money reimbursed, that makes the employee satisfaction goal
    - I can now run the analysis on the TUE Domestic Reimbursment log and can create a 3 x 2 matrix with the goal-oriented classification of the log.
- The original petri net is not an not a easy sound net. The solution is to only use one final state and not two separate onees. Thius is in ``content/TUEReimbursement/domestic_declaration_ilpn_2.pnml``, ``_ipln`` part also indicates, that is read and written by the ``I love Petri Net tool``, which requires additional XML elements added to the original ``content/TUEReimbursement/domestic_declaration.pnml`` PNML file. There is a script in the kogi-root directory ``scripts`` directory, that adds those missing elements.
- One of the traces in the paper seem to indicate that, when payment has happened, then the "(Employee) increase employee satisfaction" is satisfied, but I could not find a make connection for that quality.
- How are dependency handled when the depender is a task? The petri net still has an activity to execute that task, but according to our semantics, the task, in this case "Payment Handled by ADMIN" is automatically satisfied and does not need an activity from the petri nets. Maybe the semantics of dependency propagation needs to be changed for tasks; ie. set to unknown or pending, depending if the task was unkown or satisfied, as it would be required to execute the task again?

## Setup & Imports

In [1]:
import pandas as pd

import pm4py
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.objects.log.obj import EventLog


from Semantics.goccva_pipeline import analyse

from Ui.goccva_ui import render_from_analysis, render_all_goal_oriented_alignments_from_analysis, render_case_distribution_matrix

from Semantics.goccva_helpers import sequences_to_event_log

from Semantics.istar_processor import read_istar_model
from Semantics.petri_net_processor import read_petri_net
from Semantics.event_mapping_from_csv import read_event_mapping_csv

from pprint import pp

## Load Models

Load the goal model, process model, and mapping used by the case study.

The original petri net model cannot be used to compute alignments with pm4py as it is not proper workflow net. Thus I added the source and sink places with transitions t1, t2, and t3.

In [2]:
# Paths used in the GoCCvA repository
goal_model_path = "content/TUEReimbursement/GMjcavi2.txt"
process_model_path = "content/TUEReimbursement/domestic_declaration_ilpn_updated.pnml"
mapping_path = "content/TUEReimbursement/mappingjcavi.csv"

goal_model = read_istar_model(str(goal_model_path), qualified=True)
petri_net = read_petri_net(str(process_model_path))
activity_mapping = read_event_mapping_csv(str(mapping_path))

## Target Configuration

Define the target requirements to be evaluated. The make, break, and non-related sets are computed from the loaded goal model.

In [3]:
targets = [
    '(Admin) adequate declaration handling',
    '(Employee) Increase employee satisfaction',
]

## Target Row Computation from the Goal Model

This helper converts the make, break, and non-related sets into the rows used by the alignment visualization.

For each target and each activity in a trace:

- `M` means the activity belongs to the target make set.
- `B` means the activity belongs to the target break set.
- `NR` means the activity belongs to the target non-related set.
- `ND` is used as a fallback when the activity is not classified for the selected target.


## Handmade Test Cases

Define the traces based on the example from the paper.

In [4]:
handmade_cases = [
    {
        "requested_case": "Alignment 1",
        "trace": [
            "Declaration SUBMITTED by EMPLOYEE",
            "Declaration REJECTED by ADMINISTRATION",
            "t_tau_rev",
            "Declaration SUBMITTED by EMPLOYEE",
            "Declaration APPROVED by ADMINISTRATION",
            "Declaration APPROVED by BUDGET OWNER",
            "Declaration FINAL_APPROVED by SUPERVISOR",
            "Request Payment",
            "Payment Handled",
        ],
        "why": "ds ra ds aa ba as rp ph",
    },
    {
        "requested_case": "Alignment 2",
        "trace": [
            "Declaration SUBMITTED by EMPLOYEE",
            "Declaration APPROVED by ADMINISTRATION",
            "Declaration APPROVED by BUDGET OWNER",
            "Declaration REJECTED by BUDGET OWNER",
            "Declaration FINAL_APPROVED by SUPERVISOR",
            "Request Payment",
            "Payment Handled",
        ],
        "why": "ds aa ba rb as rp ph",
    },
    {
        "requested_case": "Alignment 3",
        "trace": [
            "Declaration SUBMITTED by EMPLOYEE",
            "Declaration APPROVED by ADMINISTRATION",
            "Declaration APPROVED by BUDGET OWNER",
            "Request Payment",
            "Payment Handled",
        ],
        "why": "ds aa ba rp ph",
    },

]

print(f"Defined {len(handmade_cases)} test case")


Defined 3 test case


In [5]:
def handmade_cases_to_event_log(cases):
    sequences = []
    for case in cases:
        trace = case.get("trace")
        sequences.append(trace)
    return sequences_to_event_log(sequences)

handmade_event_log = handmade_cases_to_event_log(handmade_cases)
print(f"EventLog created with {len(handmade_event_log)} trace(s)")

EventLog created with 3 trace(s)


In [6]:
from Semantics.enums import ElementStatus

event_log = handmade_event_log

initial_marking = {
    "(Employee) Increase employee satisfaction": ElementStatus.SATISFIED,
}

summary, detailed, contribution_to_targets = analyse(
    goal_model,
    petri_net,
    event_log,
    targets,
    activity_mapping,
    initial_marking=initial_marking,
 )

print("\nContribution to targets")
pp(contribution_to_targets)

aligning log, completed variants ::   0%|          | 0/3 [00:00<?, ?it/s]


Contribution to targets
[{'target': '(Admin) adequate declaration handling',
  'MakeSet': '(Admin) Approve Declaration, (Admin) Handle payment, (Budget '
             'Owner) Approve Declaration, (Supervisor) Approve Declaration, '
             '(Supervisor) Request Payment',
  'BreakSet': '',
  'NRSet': '(Admin) Reject Declaration, (Budget Owner) Reject Declaration, '
           '(Employee) Reject Declaration, (Employee) Save Declaration, '
           '(Employee) Submit Declaration, (Supervisor) Reject Declaration'},
 {'target': '(Employee) Increase employee satisfaction',
  'MakeSet': '(Admin) Handle payment',
  'BreakSet': '(Admin) Reject Declaration, (Budget Owner) Reject Declaration, '
              '(Employee) Reject Declaration, (Supervisor) Reject Declaration',
  'NRSet': '(Admin) Approve Declaration, (Budget Owner) Approve Declaration, '
           '(Employee) Save Declaration, (Employee) Submit Declaration, '
           '(Supervisor) Approve Declaration, (Supervisor) Request

## Activity Abbreviations

In [7]:
activity_abbreviations = {
    "Declaration REJECTED by ADMINISTRATION": "ra",
    "Payment Handled": "ph",
    "Declaration REJECTED by BUDGET OWNER": "rb",
    "Declaration SAVED by EMPLOYEE": "dsv",
    "Declaration APPROVED by ADMINISTRATION": "aa",
    "Declaration REJECTED by EMPLOYEE": "er",
    "Request Payment": "rp",
    "Declaration SUBMITTED by EMPLOYEE": "ds",
    "Declaration APPROVED by BUDGET OWNER": "ba",
    "Declaration FINAL_APPROVED by SUPERVISOR": "as",
    "Declaration REJECTED by SUPERVISOR": "rs",
    "Declaration APPROVED by PRE_APPROVER": "pa",
    "Declaration REJECTED by PRE_APPROVER": "rpa",
    "Declaration REJECTED by MISSING": "rm",
    "t_tau_rev": "tau",
    "t1": "t1",
    "t2": "t2",
    "t3": "t3",
}

print("Activity abbreviations configured")


Activity abbreviations configured


## Render Visualization

In [8]:
render_all_goal_oriented_alignments_from_analysis(
    cases=handmade_cases,
    activity_abbreviations=activity_abbreviations,
    summary=summary,
    detailed=detailed,
    contribution_to_targets=contribution_to_targets,
    title="Goal-oriented Process Alignment Examples (GoCCvA Case Study)",
)

# Goal-oriented Process Alignment Examples (GoCCvA Case Study)

The following cases illustrate combinations of alignment class and target-fulfilment class. Each table separates the process alignment row from the target-specific interpretation rows.

In [9]:
matrix_result = render_case_distribution_matrix(
    summary=summary,
    title="TUE Reimbursement Case Distribution Matrix",
    targets=targets,
)

print(matrix_result["counts"])

{'O+': 0, 'O~': 1, 'O-': 0, 'N+': 0, 'N~': 1, 'N-': 1}


# Reading the .xes file


In [10]:
log_file_path = "content/TUEReimbursement/DomesticDeclarations.xes.gz"

full_log = log_converter.apply(pm4py.read_xes(str(log_file_path)), variant=log_converter.Variants.TO_EVENT_LOG)

print("Log file loaded")
print(len(full_log))




parsing log, completed traces ::   0%|          | 0/10357 [00:00<?, ?it/s]

Log file loaded
10357


### Check that the petri net has the correct activities that appear in the log

In [11]:

vocabulary = pm4py.get_event_attribute_values(full_log, "concept:name")

print("Vocabulary from log:")
pp(sorted(vocabulary))

print("Transitions from Petri net:")
pp(sorted([t.label for t in petri_net.net.transitions]))


Vocabulary from log:
['Declaration APPROVED by ADMINISTRATION',
 'Declaration APPROVED by BUDGET OWNER',
 'Declaration APPROVED by PRE_APPROVER',
 'Declaration FINAL_APPROVED by SUPERVISOR',
 'Declaration REJECTED by ADMINISTRATION',
 'Declaration REJECTED by BUDGET OWNER',
 'Declaration REJECTED by EMPLOYEE',
 'Declaration REJECTED by MISSING',
 'Declaration REJECTED by PRE_APPROVER',
 'Declaration REJECTED by SUPERVISOR',
 'Declaration SAVED by EMPLOYEE',
 'Declaration SUBMITTED by EMPLOYEE',
 'Payment Handled',
 'Request Payment']
Transitions from Petri net:
['Declaration APPROVED by ADMINISTRATION',
 'Declaration APPROVED by BUDGET OWNER',
 'Declaration FINAL_APPROVED by SUPERVISOR',
 'Declaration REJECTED by ADMINISTRATION',
 'Declaration REJECTED by BUDGET OWNER',
 'Declaration REJECTED by EMPLOYEE',
 'Declaration REJECTED by SUPERVISOR',
 'Declaration SAVED by EMPLOYEE',
 'Declaration SUBMITTED by EMPLOYEE',
 'Payment Handled',
 'Request Payment',
 't_tau_rev']


### Show the result of the analysis of most used types of traces.

In [12]:

traces = [[event['concept:name'] for event in trace] for trace in full_log]

from collections import Counter

traces_by_frequency = Counter(tuple(item) for item in traces)

most_used_traces_by_frequency = traces_by_frequency.most_common(10)

most_used_traces_log = sequences_to_event_log([list(item) for item, _ in most_used_traces_by_frequency])

for index, (trace, count) in enumerate(most_used_traces_by_frequency):
    print(f"Case: {index+1}: {count} traces")


initial_marking = {
    "(Employee) Increase employee satisfaction (S)": ElementStatus.SATISFIED,
}
                     
summary, detailed, contribution_to_targets = analyse(
    goal_model,
    petri_net,
    most_used_traces_log,
    targets,
    activity_mapping,
    initial_marking=initial_marking,
 )

render_from_analysis(
    activity_abbreviations=activity_abbreviations,
    summary=summary,
    detailed=detailed,
    contribution_to_targets=contribution_to_targets,
    title="Goal-oriented Process Alignment Examples (GoCCvA Case Study)",
)

Case: 1: 4566 traces
Case: 2: 2411 traces
Case: 3: 1392 traces
Case: 4: 575 traces
Case: 5: 342 traces
Case: 6: 183 traces
Case: 7: 174 traces
Case: 8: 134 traces
Case: 9: 77 traces
Case: 10: 57 traces


aligning log, completed variants ::   0%|          | 0/10 [00:00<?, ?it/s]

# Goal-oriented Process Alignment Examples (GoCCvA Case Study)

The following cases illustrate combinations of alignment class and target-fulfilment class. Each table separates the process alignment row from the target-specific interpretation rows.

### Weakly compliant scenarios
These traces should be weakly compliant; however, it says they are strongly compliant.

**Hubert: Todo: This needs proper tests for the method that computes compliance!!**

In [13]:
weakly_conpliant_log = sequences_to_event_log([list(item) for item, _ in most_used_traces_by_frequency[4:5]])

summary, detailed, contribution_to_targets = analyse(
    goal_model,
    petri_net,
    weakly_conpliant_log,
    targets,
    activity_mapping,
    initial_marking=initial_marking,
 )

render_from_analysis(
    activity_abbreviations=activity_abbreviations,
    summary=summary,
    detailed=detailed,
    contribution_to_targets=contribution_to_targets,
    title="Goal-oriented Process Alignment Examples (GoCCvA Case Study)",
)

display(pd.DataFrame(summary))



# Goal-oriented Process Alignment Examples (GoCCvA Case Study)

The following cases illustrate combinations of alignment class and target-fulfilment class. Each table separates the process alignment row from the target-specific interpretation rows.

,trace_id,trace,alignment_cost,fitness,traditional_class,goal_class
0,1,Declaration SUBMITTED by EMPLOYEE | Declaratio...,30000,0.727273,non-optimal,Non-compliant


### Show the summary of all classes of conformances for all traces in the log file. 

I am trying to figure out why there are no weak-compliant traces in the log that are optimal. 

In [14]:
summary, _, _ = analyse(
    goal_model,
    petri_net,
    full_log,
    targets,
    activity_mapping,
    initial_marking=initial_marking,
)

matrix_result = render_case_distribution_matrix(
    summary=summary,
    title="TUE Reimbursement Case Distribution Matrix",
    targets=targets,
)

print(matrix_result["counts"])

aligning log, completed variants ::   0%|          | 0/90 [00:00<?, ?it/s]

{'O+': 2411, 'O~': 0, 'O-': 185, 'N+': 1, 'N~': 314, 'N-': 7446}


### Goal and process compliance classes by unique traces

In [15]:
traces_by_frequency = Counter(tuple(item) for item in traces)

all_traces_by_frequency = traces_by_frequency.most_common()

all_categories_log = sequences_to_event_log([list(item) for item, _ in all_traces_by_frequency])

summary, _, _ = analyse(
    goal_model,
    petri_net,
    all_categories_log,
    targets,
    activity_mapping,
    initial_marking=initial_marking,
)

matrix_result = render_case_distribution_matrix(
    summary=summary,
    title="TUE Reimbursement Case Distribution Matrix",
    targets=targets,
)

print(matrix_result["counts"])

aligning log, completed variants ::   0%|          | 0/90 [00:00<?, ?it/s]

{'O+': 1, 'O~': 0, 'O-': 3, 'N+': 1, 'N~': 19, 'N-': 66}


In [16]:
needle = "REJECTED"


def _trace_contains_activity(trace, pattern: str) -> bool:
    p = pattern.upper()

    items = trace
    if isinstance(trace, tuple) and len(trace) == 2 and isinstance(trace[0], (list, tuple)):
        items = trace[0]

    return any(p in str(activity).upper() for activity in items)


traces_containing_rejected = [
    trace
    for trace, _ in traces_by_frequency.items()
    if _trace_contains_activity(trace, needle)
]

print(f"Found {len(traces_containing_rejected)} unique traces containing '{needle}'")

summary, _, _ = analyse(
    goal_model,
    petri_net,
    sequences_to_event_log(traces_containing_rejected),
    targets,
    activity_mapping,
)

matrix_result = render_case_distribution_matrix(
    summary=summary,
    title="TUE Reimbursement Case Distribution Matrix",
    targets=targets,
)

print(matrix_result["counts"])

Found 84 unique traces containing 'REJECTED'


aligning log, completed variants ::   0%|          | 0/84 [00:00<?, ?it/s]

{'O+': 0, 'O~': 0, 'O-': 3, 'N+': 1, 'N~': 19, 'N-': 61}
